# Trade Corridor Health Monitor — research notebook

**Execution:** run from the repository root, with the six input CSVs in `data/raw/`. The original notebook was a course deliverable; this repository version adds year-grouped tuning folds and input-path configuration. Previously saved outputs are historical and **must be regenerated** after obtaining the input files.

The published dashboard is a snapshot, not a live forecast service. See `docs/METHODOLOGY.md` and `docs/LIMITATIONS.md`.


In [ ]:
# ===============================================================
# BEYOND HISTORICAL DISRUPTIONS:
# A Predictive Decision Framework for Monitoring
# Global Trade Corridor Health
#
# Version 2.0
# Block 1A
# ===============================================================

# ===============================================================
# IMPORT LIBRARIES
# ===============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

sns.set_theme(
    style="whitegrid",
    context="talk"
)

print("Libraries imported successfully.")


# ===============================================================
# PROJECT CONFIGURATION
# ===============================================================

DATA_PATH = Path("data/raw")
OUTPUT_PATH = Path("data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

PROJECT_TITLE = (
    "Beyond Historical Disruptions: "
    "A Predictive Decision Framework for "
    "Monitoring Global Trade Corridor Health"
)

print("=" * 80)
print(PROJECT_TITLE)
print("=" * 80)


# ===============================================================
# LOAD DATASETS
# ===============================================================

trade = pd.read_csv(DATA_PATH / "trade_flows.csv")

shipping = pd.read_csv(DATA_PATH / "shipping_rates.csv")

tariff = pd.read_csv(DATA_PATH / "tariff_timeline.csv")

commodity = pd.read_csv(DATA_PATH / "commodity_prices_supply_chain.csv")

disruption = pd.read_csv(DATA_PATH / "disruption_events.csv")

industry = pd.read_csv(DATA_PATH / "industry_exposure.csv")

print("Datasets loaded successfully.\n")


# ===============================================================
# DATASET OVERVIEW
# ===============================================================

datasets = {
    "Trade Flows": trade,
    "Shipping Rates": shipping,
    "Tariff Timeline": tariff,
    "Commodity Prices": commodity,
    "Disruption Events": disruption,
    "Industry Exposure": industry
}

overview = []

for name, df in datasets.items():

    overview.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values":
            int(df.isna().sum().sum()),

        "Duplicate Rows":
            int(df.duplicated().sum())

    })

overview = pd.DataFrame(overview)

display(overview)


# ===============================================================
# TRADE DATA OVERVIEW
# ===============================================================

print("=" * 80)
print("TRADE DATA")
print("=" * 80)

print("\nShape")

print(trade.shape)

print("\nColumns")

print(list(trade.columns))

print("\nPreview")

display(trade.head())


# ===============================================================
# UNIQUE VALUES
# ===============================================================

print("=" * 80)
print("UNIQUE EXPORTERS")
print("=" * 80)

print(sorted(trade["exporter"].unique()))

print()

print("=" * 80)
print("UNIQUE IMPORTERS")
print("=" * 80)

print(sorted(trade["importer"].unique()))

print()

print("=" * 80)
print("TRADE CATEGORIES")
print("=" * 80)

print(sorted(trade["trade_category"].unique()))

print()

print("=" * 80)
print("KEY GOODS")
print("=" * 80)

print(sorted(trade["key_goods"].unique()))


# ===============================================================
# ENTITY CLASSIFICATION
# ===============================================================

regional_blocs = [
    "EU"
]

trade["exporter_entity_type"] = np.where(

    trade["exporter"].isin(regional_blocs),

    "Regional Bloc",

    "Country"

)

trade["importer_entity_type"] = np.where(

    trade["importer"].isin(regional_blocs),

    "Regional Bloc",

    "Country"

)


# ===============================================================
# CREATE CORRIDOR
# ===============================================================

trade["corridor"] = (

    trade["exporter"]

    + " → "

    + trade["importer"]

)


# ===============================================================
# BASIC VALIDATION
# ===============================================================

print("=" * 80)
print("VALIDATION")
print("=" * 80)

print(
    "Years:",
    trade["year"].min(),
    "-",
    trade["year"].max()
)

print(
    "Unique Corridors:",
    trade["corridor"].nunique()
)

print(
    "Exporters:",
    trade["exporter"].nunique()
)

print(
    "Importers:",
    trade["importer"].nunique()
)

print(
    "Trade Categories:",
    trade["trade_category"].nunique()
)

print(
    "Key Goods:",
    trade["key_goods"].nunique()
)

print(
    "Regional Bloc Exporters:",
    (
        trade["exporter_entity_type"]
        == "Regional Bloc"
    ).sum()
)

print(
    "Regional Bloc Importers:",
    (
        trade["importer_entity_type"]
        == "Regional Bloc"
    ).sum()
)


# ===============================================================
# PANEL STRUCTURE CHECK
# ===============================================================

panel = (

    trade

    .groupby("corridor")

    .agg(

        first_year=("year", "min"),

        last_year=("year", "max"),

        observations=("year", "count")

    )

    .reset_index()

)

display(panel.head(15))


print("=" * 80)
print("OBSERVATIONS PER CORRIDOR")
print("=" * 80)

display(

    panel["observations"]

    .value_counts()

    .sort_index()

)


# ===============================================================
# CHECK FOR MISSING YEARS
# ===============================================================

expected_years = set(

    range(

        trade["year"].min(),

        trade["year"].max() + 1

    )

)

missing_summary = []

for corridor, group in trade.groupby("corridor"):

    years = set(group["year"])

    missing = sorted(

        expected_years - years

    )

    missing_summary.append({

        "corridor": corridor,

        "missing_years": len(missing),

        "years": missing

    })

missing_summary = pd.DataFrame(

    missing_summary

)

print("=" * 80)
print("CORRIDORS WITH MISSING YEARS")
print("=" * 80)

display(

    missing_summary.loc[

        missing_summary["missing_years"] > 0

    ]

)

# ===============================================================
# BLOCK 1B
# TRADE SECTOR STANDARDIZATION
# ===============================================================

print("=" * 80)
print("BLOCK 1B - TRADE SECTOR STANDARDIZATION")
print("=" * 80)


# --------------------------------------------------------------
# CLEAN TEXT COLUMNS
# --------------------------------------------------------------

text_columns = [

    "exporter",
    "importer",
    "trade_category",
    "key_goods"

]

for col in text_columns:

    trade[col] = (

        trade[col]

        .astype(str)

        .str.strip()

        .str.lower()

    )


# Restore title case for exporter/importer

trade["exporter"] = trade["exporter"].str.title()

trade["importer"] = trade["importer"].str.title()

trade["exporter"] = trade["exporter"].replace(
    {"Eu":"EU","Usa":"USA","Uk":"UK","Southkorea":"SouthKorea","Uae":"UAE"}
)

trade["importer"] = trade["importer"].replace(
    {"Eu":"EU","Usa":"USA","Uk":"UK","Southkorea":"SouthKorea","Uae":"UAE"}
)

trade["corridor"] = (

    trade["exporter"]

    + " → "

    + trade["importer"]

)


print("✓ Text cleaning completed")


# --------------------------------------------------------------
# STANDARD SECTOR DEFINITIONS
# --------------------------------------------------------------

sector_keywords = {

    "Energy":[

        "oil",
        "crude",
        "gas",
        "lng",
        "petroleum",
        "petrochemical",
        "energy"

    ],

    "Mining & Metals":[

        "iron ore",
        "iron",
        "coal",
        "copper",
        "gold",
        "silver",
        "zinc",
        "nickel",
        "lithium",
        "platinum",
        "phosphate",
        "minerals",
        "mineral",
        "steel"

    ],

    "Electronics & Semiconductors":[

        "electronics",
        "electronic",
        "chip",
        "chips",
        "semiconductor",
        "semiconductors"

    ],

    "Automotive & Machinery":[

        "vehicle",
        "vehicles",
        "machinery",
        "machine",
        "aircraft",
        "aerospace"

    ],

    "Pharmaceuticals & Chemicals":[

        "pharma",
        "pharmaceutical",
        "chemical",
        "chemicals",
        "medical"

    ],

    "Agriculture & Food":[

        "soybeans",
        "soybean",
        "beef",
        "food",
        "coffee",
        "fruit",
        "grain",
        "rice",
        "corn",
        "livestock",
        "agriculture",
        "agricultural",
        "palm oil",
        "palm"

    ],

    "Textiles & Apparel":[

        "textiles",
        "textile",
        "garments",
        "garment",
        "knitwear",
        "footwear",
        "cotton",
        "apparel"

    ],

    "Consumer & Luxury Goods":[

        "luxury",
        "consumer"

    ]

}

print("✓ Sector dictionary created")

# ===============================================================
# WEIGHTED SECTOR CLASSIFIER
# ===============================================================

def classify_trade_sector(goods):

    goods = str(goods).lower()

    # -----------------------------------------
    # Split into individual products
    # -----------------------------------------

    product_list = [

        product.strip()

        for product in goods.split(",")

        if product.strip() != ""

    ]


    # -----------------------------------------
    # Initialize sector scores
    # -----------------------------------------

    sector_scores = {

        sector: 0

        for sector in sector_keywords.keys()

    }


    # -----------------------------------------
    # Score every product
    # -----------------------------------------

    for product in product_list:

        for sector, keywords in sector_keywords.items():

            for keyword in keywords:

                if keyword in product:

                    sector_scores[sector] += 1


    # -----------------------------------------
    # No matches
    # -----------------------------------------

    if max(sector_scores.values()) == 0:

        return pd.Series({

            "primary_sector":"Multi-sector Trade",

            "secondary_sector":"None",

            "sector_confidence":0,

            "sector_diversity":0,

            "matched_keywords":0

        })


    # -----------------------------------------
    # Sort scores
    # -----------------------------------------

    ranked = sorted(

        sector_scores.items(),

        key=lambda x: x[1],

        reverse=True

    )


    primary_sector = ranked[0][0]

    primary_score = ranked[0][1]


    if len(ranked) > 1:

        secondary_sector = ranked[1][0]

        secondary_score = ranked[1][1]

    else:

        secondary_sector = "None"

        secondary_score = 0


    total_matches = sum(

        sector_scores.values()

    )


    confidence = round(

        primary_score /

        total_matches,

        3

    )


    diversity = (

        len(

            [

                score

                for score in sector_scores.values()

                if score > 0

            ]

        )

    )


    return pd.Series({

        "primary_sector":primary_sector,

        "secondary_sector":secondary_sector,

        "sector_confidence":confidence,

        "sector_diversity":diversity,

        "matched_keywords":total_matches

    })


# ===============================================================
# APPLY CLASSIFIER
# ===============================================================

sector_results = (

    trade["key_goods"]

    .apply(classify_trade_sector)

)

trade = pd.concat(

    [

        trade,

        sector_results

    ],

    axis=1

)


trade.rename(

    columns={

        "primary_sector":"standard_trade_category"

    },

    inplace=True

)


print("✓ Sector classification completed")


# ===============================================================
# TRADE BASKET FEATURES
# ===============================================================

trade["number_of_products"] = (

    trade["key_goods"]

    .str.split(",")

    .str.len()

)


trade["trade_basket_type"] = np.where(

    trade["number_of_products"] >= 3,

    "Diversified",

    "Specialized"

)


print("✓ Trade basket features created")


# ===============================================================
# ENTITY LABELS
# ===============================================================

trade["exporter_entity"] = np.where(

    trade["exporter"] == "EU",

    "Regional Trade Bloc",

    "Sovereign Country"

)


trade["importer_entity"] = np.where(

    trade["importer"] == "EU",

    "Regional Trade Bloc",

    "Sovereign Country"

)


print("✓ Entity classification completed")

# --------------------------------------------------------------
# Replace weak classifications with Multi-sector Trade
# --------------------------------------------------------------

trade["standard_trade_category"] = np.where(
    (trade["sector_confidence"] < 0.60) &
    (trade["sector_diversity"] >= 3),
    "Multi-sector Trade",
    trade["standard_trade_category"]
)

# ===============================================================
# BLOCK 2A
# DATA UNDERSTANDING BEFORE MERGING
# ===============================================================

print("="*80)
print("BLOCK 2A - UNDERSTANDING ALL DATASETS")
print("="*80)


datasets = {

    "Trade": trade,

    "Shipping": shipping,

    "Tariff": tariff,

    "Commodity": commodity,

    "Disruption": disruption,

    "Industry": industry

}


for name, df in datasets.items():

    print("\n")
    print("="*80)
    print(name.upper())
    print("="*80)

    print("\nShape")

    print(df.shape)

    print("\nColumns")

    print(df.columns.tolist())

    print("\nMissing Values")

    display(df.isna().sum())

    print("\nPreview")

    display(df.head())

# ===============================================================
# UNIQUE VALUES OF IMPORTANT VARIABLES
# ===============================================================

important_columns = [

    "year",

    "exporter",

    "importer",

    "commodity",

    "country",

    "trade_category",

    "industry",

    "corridor"

]


for name, df in datasets.items():

    print("\n")
    print("="*80)

    print(name)

    print("="*80)

    for col in important_columns:

        if col in df.columns:

            print(f"\n{col}")

            print(df[col].unique()[:20])

            print("Unique:",df[col].nunique())

# ===============================================================
# DUPLICATE CHECK
# ===============================================================

for name, df in datasets.items():

    print("\n")
    print("="*80)

    print(name)

    print("="*80)

    print(

        "Duplicate Rows:",

        df.duplicated().sum()

    )

# ===============================================================
# YEAR COVERAGE
# ===============================================================

for name, df in datasets.items():

    if "year" in df.columns:

        print(

            name,

            df.year.min(),

            "->",

            df.year.max()

        )

trade['yoy_growth_pct'].isna().sum()

# ===============================================================
# BLOCK 2B
# BUILD YEARLY FEATURE TABLES
# ===============================================================

print("="*80)
print("BLOCK 2B - BUILD YEARLY FEATURE TABLES")
print("="*80)

# ===============================================================
# SHIPPING FEATURES
# ===============================================================

shipping_yearly = (

    shipping

    .groupby("year", as_index=False)

    .agg(

        avg_bdi=("baltic_dry_index","mean"),

        max_bdi=("baltic_dry_index","max"),

        avg_container_rate=("container_rate_usd_40ft","mean"),

        max_container_rate=("container_rate_usd_40ft","max"),

        avg_air_rate=("air_cargo_rate_usd_kg","mean"),

        avg_supply_chain_pressure=("supply_chain_pressure_index","mean"),

        max_supply_chain_pressure=("supply_chain_pressure_index","max"),

        avg_on_time_delivery=("on_time_delivery_pct","mean")

    )

)

print("Shipping yearly shape")

print(shipping_yearly.shape)

display(shipping_yearly.head())

# ===============================================================
# DISRUPTION FEATURES
# ===============================================================

severity_mapping = {

    "low":1,

    "medium":2,

    "high":3

}

disruption["severity_score"] = (

    disruption["severity"]

    .map(severity_mapping)

)

disruption_yearly = (

    disruption

    .groupby("year",as_index=False)

    .agg(

        disruption_events=("event_name","count"),

        avg_duration=("duration_days","mean"),

        max_severity=("severity_score","max"),

        avg_trade_impact=("trade_volume_impact_pct","mean"),

        avg_recovery=("recovery_months","mean"),

        pandemic_events=("is_pandemic","sum"),

        geopolitical_events=("is_geopolitical","sum"),

        natural_events=("is_natural","sum"),

        financial_events=("is_financial","sum")

    )

)

print("Disruption yearly shape")

print(disruption_yearly.shape)

display(disruption_yearly.head())

# ===============================================================
# COMMODITY YEARLY FEATURES
# ===============================================================

print("=" * 80)
print("COMMODITY YEARLY FEATURES")
print("=" * 80)


# ---------------------------------------------------------------
# Average annual price by commodity category
# ---------------------------------------------------------------

commodity_category_yearly = (
    commodity
    .groupby(
        [
            "year",
            "category"
        ],
        as_index=False
    )
    .agg(
        average_price=(
            "price",
            "mean"
        )
    )
)


# ---------------------------------------------------------------
# Convert categories into separate columns
# ---------------------------------------------------------------

commodity_yearly = (
    commodity_category_yearly
    .pivot(
        index="year",
        columns="category",
        values="average_price"
    )
    .reset_index()
)


# ---------------------------------------------------------------
# Clean column names
# ---------------------------------------------------------------

commodity_yearly.columns = [
    str(column)
    .strip()
    .lower()
    .replace(" ", "_")
    .replace("&", "and")
    for column in commodity_yearly.columns
]


# ---------------------------------------------------------------
# Rename commodity-category columns
# ---------------------------------------------------------------

commodity_yearly = commodity_yearly.rename(
    columns={
        "industrial_metal": "industrial_metal_price_index",
        "precious_metal": "precious_metal_price_index",
        "energy": "energy_price_index",
        "agriculture": "agriculture_price_index"
    }
)


# ---------------------------------------------------------------
# Create yearly percentage changes
# ---------------------------------------------------------------

commodity_price_columns = [
    column
    for column in commodity_yearly.columns
    if column != "year"
]

for column in commodity_price_columns:

    commodity_yearly[
        f"{column}_change_pct"
    ] = (
        commodity_yearly[column]
        .pct_change()
        .mul(100)
    )


print(
    "Commodity yearly shape:",
    commodity_yearly.shape
)

display(
    commodity_yearly.head()
)

print(
    "Commodity years:",
    commodity_yearly["year"].min(),
    "-",
    commodity_yearly["year"].max()
)

# ===============================================================
# INDUSTRY EXPOSURE MAPPING
# ===============================================================

print("=" * 80)
print("INDUSTRY EXPOSURE MAPPING")
print("=" * 80)


# ---------------------------------------------------------------
# Map standardized trade sectors to the closest industry
# ---------------------------------------------------------------

sector_to_industry = {

    "Electronics & Semiconductors":
        "Semiconductors",

    "Automotive & Machinery":
        "Automotive",

    "Pharmaceuticals & Chemicals":
        "Pharma & Medical",

    "Energy":
        "Energy & Chemicals",

    "Mining & Metals":
        "Energy & Chemicals",

    "Agriculture & Food":
        "Food & Agriculture",

    "Textiles & Apparel":
        "Retail & Apparel",

    "Consumer & Luxury Goods":
        "Luxury Goods"

}


trade["mapped_industry"] = (
    trade["standard_trade_category"]
    .map(sector_to_industry)
)


# ---------------------------------------------------------------
# Keep a focused set of industry variables
# ---------------------------------------------------------------

industry_features = industry[
    [
        "year",
        "industry",
        "overall_vulnerability",
        "inventory_days",
        "supplier_concentration_hhi",
        "just_in_time_dependency",
        "nearshoring_score_2024"
    ]
].copy()


# ---------------------------------------------------------------
# Create annual averages for multi-sector corridors
# ---------------------------------------------------------------

industry_average_yearly = (
    industry_features
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        overall_vulnerability=(
            "overall_vulnerability",
            "mean"
        ),
        inventory_days=(
            "inventory_days",
            "mean"
        ),
        supplier_concentration_hhi=(
            "supplier_concentration_hhi",
            "mean"
        ),
        just_in_time_dependency=(
            "just_in_time_dependency",
            "mean"
        ),
        nearshoring_score_2024=(
            "nearshoring_score_2024",
            "mean"
        )
    )
)


industry_average_yearly[
    "industry"
] = "Multi-sector Average"


# ---------------------------------------------------------------
# Assign multi-sector label
# ---------------------------------------------------------------

trade["mapped_industry"] = (
    trade["mapped_industry"]
    .fillna(
        "Multi-sector Average"
    )
)


# ---------------------------------------------------------------
# Combine original industries and multi-sector averages
# ---------------------------------------------------------------

industry_clean = pd.concat(
    [
        industry_features,
        industry_average_yearly[
            industry_features.columns
        ]
    ],
    ignore_index=True
)


print(
    "Industry clean shape:",
    industry_clean.shape
)

print(
    "Mapped industries:"
)

display(
    trade[
        [
            "standard_trade_category",
            "mapped_industry"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "standard_trade_category"
    )
)

# ===============================================================
# MASTER ANALYTICAL DATASET
# ===============================================================

print("=" * 80)
print("BUILDING MASTER ANALYTICAL DATASET")
print("=" * 80)


# ---------------------------------------------------------------
# Start from the 1,250-row trade corridor panel
# ---------------------------------------------------------------

master = trade.copy()

initial_rows = master.shape[0]

print(
    "Starting rows:",
    initial_rows
)


# ---------------------------------------------------------------
# Merge yearly shipping features
# ---------------------------------------------------------------

master = master.merge(
    shipping_yearly,
    on="year",
    how="left",
    validate="many_to_one"
)

print(
    "After shipping merge:",
    master.shape
)

# ---------------------------------------------------------------
# Merge yearly disruption features
# ---------------------------------------------------------------

master = master.merge(
    disruption_yearly,
    on="year",
    how="left",
    validate="many_to_one"
)

print(
    "After disruption merge:",
    master.shape
)


# ---------------------------------------------------------------
# Merge yearly commodity-price features
# ---------------------------------------------------------------

master = master.merge(
    commodity_yearly,
    on="year",
    how="left",
    validate="many_to_one"
)

print(
    "After commodity merge:",
    master.shape
)


# ---------------------------------------------------------------
# Merge industry-exposure features
# ---------------------------------------------------------------

master = master.merge(
    industry_clean,
    left_on=[
        "year",
        "mapped_industry"
    ],
    right_on=[
        "year",
        "industry"
    ],
    how="left",
    validate="many_to_one"
)

print(
    "After industry merge:",
    master.shape
)


# ---------------------------------------------------------------
# Drop duplicated industry label
# ---------------------------------------------------------------

master = master.drop(
    columns=[
        "industry"
    ]
)


# ===============================================================
# HANDLE STRUCTURAL MISSING VALUES
# ===============================================================

# Disruption-event data only exists in years with recorded events.
# Event counts and event-type indicators can therefore be set to zero.

disruption_count_columns = [
    "disruption_events",
    "pandemic_events",
    "geopolitical_events",
    "natural_events",
    "financial_events"
]

for column in disruption_count_columns:

    if column in master.columns:

        master[column] = (
            master[column]
            .fillna(0)
        )


# Non-event disruption measures remain zero in years
# without recorded disruptions.

disruption_measure_columns = [
    "avg_duration",
    "max_severity",
    "avg_trade_impact",
    "avg_recovery"
]

for column in disruption_measure_columns:

    if column in master.columns:

        master[column] = (
            master[column]
            .fillna(0)
        )


# Commodity prices begin in 2010.
# Do not fabricate values for 2000–2009.
# These missing values will be handled later during model preparation.


# ===============================================================
# SORT MASTER PANEL
# ===============================================================

master = (
    master
    .sort_values(
        [
            "corridor",
            "year"
        ]
    )
    .reset_index(drop=True)
)


# ===============================================================
# VALIDATE PANEL INTEGRITY
# ===============================================================

assert master.shape[0] == initial_rows, (
    "Row count changed during merging."
)

assert master[
    [
        "corridor",
        "year"
    ]
].duplicated().sum() == 0, (
    "Duplicate corridor-year observations were created."
)

assert master["corridor"].nunique() == 50, (
    "Expected 50 unique corridors."
)

assert master["year"].min() == 2000, (
    "Unexpected first year."
)

assert master["year"].max() == 2024, (
    "Unexpected final year."
)


print("\n" + "=" * 80)
print("MASTER DATASET VALIDATION")
print("=" * 80)

print(
    "Rows:",
    master.shape[0]
)

print(
    "Columns:",
    master.shape[1]
)

print(
    "Unique corridors:",
    master["corridor"].nunique()
)

print(
    "Years:",
    master["year"].min(),
    "-",
    master["year"].max()
)

print(
    "Duplicate corridor-years:",
    master[
        [
            "corridor",
            "year"
        ]
    ].duplicated().sum()
)


# ===============================================================
# MISSING-VALUE SUMMARY
# ===============================================================

missing_summary_master = (
    master
    .isna()
    .sum()
    .loc[
        lambda values:
        values > 0
    ]
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_values"
    )
    .to_frame()
)

missing_summary_master[
    "missing_pct"
] = (
    missing_summary_master[
        "missing_values"
    ]
    .div(
        len(master)
    )
    .mul(100)
    .round(2)
)

print("\nColumns with missing values:")

display(
    missing_summary_master
)


# ===============================================================
# PREVIEW
# ===============================================================

preview_columns = [
    "year",
    "corridor",
    "standard_trade_category",
    "mapped_industry",
    "trade_value_bn_usd",
    "yoy_growth_pct",
    "effective_tariff_rate_pct",
    "concentration_risk",
    "avg_supply_chain_pressure",
    "disruption_events",
    "overall_vulnerability"
]

preview_columns = [
    column
    for column in preview_columns
    if column in master.columns
]

display(
    master[
        preview_columns
    ].head(15)
)


# ===============================================================
# EXPORT MASTER DATASET
# ===============================================================

master.to_csv(
    OUTPUT_PATH / "master_trade_corridor_dataset.csv",
    index=False
)

print(
    "\nmaster_trade_corridor_dataset.csv exported successfully."
)

# ===============================================================
# BLOCK 3A
# TIME-SERIES FEATURE ENGINEERING
# ===============================================================

print("=" * 80)
print("BLOCK 3A - FEATURE ENGINEERING")
print("=" * 80)

master = master.sort_values(
    ["corridor", "year"]
).reset_index(drop=True)


# ===============================================================
# LAG FEATURES
# ===============================================================

master["trade_value_lag1"] = (
    master
    .groupby("corridor")["trade_value_bn_usd"]
    .shift(1)
)

master["growth_lag1"] = (
    master
    .groupby("corridor")["yoy_growth_pct"]
    .shift(1)
)


# ===============================================================
# ROLLING GROWTH
# ===============================================================

master["growth_3yr_avg"] = (

    master

    .groupby("corridor")["yoy_growth_pct"]

    .transform(

        lambda x:

        x.rolling(
            window=3,
            min_periods=1
        ).mean()

    )

)


# ===============================================================
# TRADE VOLATILITY
# ===============================================================

master["growth_3yr_volatility"] = (

    master

    .groupby("corridor")["yoy_growth_pct"]

    .transform(

        lambda x:

        x.rolling(
            window=3,
            min_periods=2
        ).std()

    )

)


# ===============================================================
# TRADE MOMENTUM
# ===============================================================

historical_growth_baseline = (

    master

    .groupby("corridor")["yoy_growth_pct"]

    .transform(

        lambda x:

        x.shift(2)

         .rolling(
             window=3,
             min_periods=3
         )

         .mean()

    )

)

master["trade_momentum"] = (

    master["growth_lag1"]

    -

    historical_growth_baseline

)

print("✓ Time-series features created")

# ===============================================================
# MOMENTUM VALIDATION CHECK
# ===============================================================

# 1. Summary statistics
print("\nTrade Momentum Summary:")
display(
    master["trade_momentum"].describe()
)


# 2. Inspect one corridor
print("\nSample Corridor Momentum:")

sample_corridor = (
    master["corridor"]
    .value_counts()
    .idxmax()
)

print("Corridor:", sample_corridor)

display(
    master.loc[
        master["corridor"] == sample_corridor,
        [
            "year",
            "yoy_growth_pct",
            "growth_lag1",
            "growth_3yr_avg",
            "trade_momentum"
        ]
    ]
)


# 3. Momentum distribution
plt.figure(figsize=(9, 5))

master["trade_momentum"].hist(
    bins=30
)

plt.title("Distribution of Trade Momentum")
plt.xlabel("Trade Momentum")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# ===============================================================
# SHARE OF GLOBAL TRADE
# ===============================================================

annual_trade = (

    master

    .groupby("year")["trade_value_bn_usd"]

    .transform("sum")

)

master["share_global_trade"] = (

    master["trade_value_bn_usd"]

    /

    annual_trade

)


# ===============================================================
# TRADE RANK
# ===============================================================

master["trade_rank"] = (

    master

    .groupby("year")["trade_value_bn_usd"]

    .rank(

        ascending=False,

        method="dense"

    )

)


print("✓ Corridor importance features created")

# ===============================================================
# SUPPLY CHAIN PRESSURE CHANGE
# ===============================================================

master["supply_chain_pressure_change"] = (
    master
    .groupby("corridor")["avg_supply_chain_pressure"]
    .diff()
)


# ===============================================================
# DISRUPTION INTENSITY
# ===============================================================

master["disruption_intensity"] = (

    master["disruption_events"]

    *

    master["max_severity"]

)


print("✓ Risk features created")

# ===============================================================
# NEXT-YEAR TARGET
# ===============================================================

master["target_next_year_growth"] = (

    master

    .groupby("corridor")["yoy_growth_pct"]

    .shift(-1)

)


print("✓ Prediction target created")

# ===============================================================
# VALIDATION
# ===============================================================

print()

print(master.shape)

print()

master_features = [

    "trade_value_lag1",

    "growth_3yr_avg",

    "growth_3yr_volatility",

    "trade_momentum",

    "share_global_trade",

    "trade_rank",

    "supply_chain_pressure_change",

    "disruption_intensity",

    "target_next_year_growth"

]

display(

    master[master_features]

    .describe()

)

display(

    master.head()

)

# ===============================================================
# CHECK NEW FEATURES
# ===============================================================

new_features = [
    "trade_value_lag1",
    "growth_3yr_avg",
    "growth_3yr_volatility",
    "trade_momentum",
    "share_global_trade",
    "trade_rank",
    "supply_chain_pressure_change",
    "disruption_intensity",
    "target_next_year_growth"
]

print(master[new_features].isna().sum())

display(master[new_features].describe())

# ===============================================================
# BLOCK 4A
# GLOBAL TRADE LANDSCAPE
# ===============================================================

import matplotlib.pyplot as plt

print("=" * 80)
print("BLOCK 4A - GLOBAL TRADE LANDSCAPE")
print("=" * 80)


annual_trade = (

    master

    .groupby("year", as_index=False)

    .agg(

        total_trade=("trade_value_bn_usd","sum")

    )

)


plt.figure(figsize=(12,6))

plt.plot(

    annual_trade["year"],

    annual_trade["total_trade"],

    linewidth=3

)

plt.title("Global Trade Value (2000–2024)", fontsize=16)

plt.xlabel("Year")

plt.ylabel("Trade Value (Bn USD)")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

# ===============================================================
# TOP TRADE CORRIDORS
# ===============================================================

top_corridors = (

    master

    .groupby("corridor",as_index=False)

    .agg(

        avg_trade=("trade_value_bn_usd","mean")

    )

    .sort_values(

        "avg_trade",

        ascending=False

    )

    .head(15)

)


plt.figure(figsize=(12,7))

plt.barh(

    top_corridors["corridor"],

    top_corridors["avg_trade"]

)

plt.title(

    "Top 15 Trade Corridors by Average Trade Value",

    fontsize=15

)

plt.xlabel("Average Trade Value (Bn USD)")

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

# ===============================================================
# TRADE BY SECTOR
# ===============================================================

sector_trade = (

    master

    .groupby(

        "standard_trade_category",

        as_index=False

    )

    .agg(

        total_trade=("trade_value_bn_usd","sum")

    )

    .sort_values(

        "total_trade",

        ascending=False

    )

)


plt.figure(figsize=(11,6))

plt.bar(

    sector_trade["standard_trade_category"],

    sector_trade["total_trade"]

)

plt.xticks(rotation=45, ha="right")

plt.ylabel("Trade Value (Bn USD)")

plt.title(

    "Global Trade by Standardized Sector",

    fontsize=15

)

plt.tight_layout()

plt.show()

# ===============================================================
# CHART 4 - EXPORTER TO IMPORTER TRADE HEATMAP
# ===============================================================

# Aggregate average trade value for each exporter-importer pair
trade_heatmap = (
    master
    .groupby(
        [
            "exporter",
            "importer"
        ],
        as_index=False
    )
    .agg(
        avg_trade_value=(
            "trade_value_bn_usd",
            "mean"
        )
    )
)

# Convert the table into heatmap format
trade_heatmap_matrix = (
    trade_heatmap
    .pivot(
        index="exporter",
        columns="importer",
        values="avg_trade_value"
    )
    .fillna(0)
)

plt.figure(figsize=(13, 11))

plt.imshow(
    trade_heatmap_matrix,
    aspect="auto"
)

plt.colorbar(
    label="Average Trade Value (Bn USD)"
)

plt.xticks(
    ticks=range(
        len(trade_heatmap_matrix.columns)
    ),
    labels=trade_heatmap_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=range(
        len(trade_heatmap_matrix.index)
    ),
    labels=trade_heatmap_matrix.index
)

plt.title(
    "Average Trade Value by Exporter–Importer Relationship",
    fontsize=15
)

plt.xlabel("Importer")

plt.ylabel("Exporter")

plt.tight_layout()

plt.show()

# ===============================================================
# EXECUTIVE SUMMARY
# ===============================================================

summary = {

    "Years": master["year"].nunique(),

    "Corridors": master["corridor"].nunique(),

    "Exporters": master["exporter"].nunique(),

    "Importers": master["importer"].nunique(),

    "Trade Sectors": master["standard_trade_category"].nunique(),

    "Total Trade (Bn USD)": round(master["trade_value_bn_usd"].sum(),2),

    "Average Annual Trade": round(annual_trade["total_trade"].mean(),2)

}

summary = pd.DataFrame(

    summary.items(),

    columns=["Metric","Value"]

)

display(summary)

# ===============================================================
# BLOCK 4B
# CORRIDOR PERFORMANCE AND STABILITY
# ===============================================================

print("=" * 80)
print("BLOCK 4B - CORRIDOR PERFORMANCE AND STABILITY")
print("=" * 80)


# ===============================================================
# TRADE GROWTH DISTRIBUTION
# ===============================================================

growth_data = (
    master["yoy_growth_pct"]
    .dropna()
)

plt.figure(figsize=(11, 6))

plt.hist(
    growth_data,
    bins=30,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(
    growth_data.mean(),
    linestyle="--",
    linewidth=2,
    label=(
        f"Mean Growth = "
        f"{growth_data.mean():.2f}%"
    )
)

plt.axvline(
    0,
    linewidth=1
)

plt.title(
    "Distribution of Annual Trade Growth Across Corridors",
    fontsize=15
)

plt.xlabel("Year-over-Year Trade Growth (%)")

plt.ylabel("Number of Corridor-Year Observations")

plt.legend()

plt.tight_layout()

plt.show()

# ===============================================================
# CORRIDOR MOMENTUM VS VOLATILITY
# ===============================================================

corridor_performance = (
    master
    .groupby(
        "corridor",
        as_index=False
    )
    .agg(
        avg_growth=(
            "yoy_growth_pct",
            "mean"
        ),
        avg_momentum=(
            "trade_momentum",
            "mean"
        ),
        avg_volatility=(
            "growth_3yr_volatility",
            "mean"
        ),
        avg_trade_value=(
            "trade_value_bn_usd",
            "mean"
        )
    )
    .dropna(
        subset=[
            "avg_momentum",
            "avg_volatility"
        ]
    )
)


plt.figure(figsize=(12, 7))

plt.scatter(
    corridor_performance[
        "avg_volatility"
    ],
    corridor_performance[
        "avg_momentum"
    ],
    s=(
        corridor_performance[
            "avg_trade_value"
        ]
        / corridor_performance[
            "avg_trade_value"
        ].max()
        * 900
        + 40
    ),
    alpha=0.7
)

plt.axhline(
    0,
    linewidth=1,
    linestyle="--"
)

plt.axvline(
    corridor_performance[
        "avg_volatility"
    ].median(),
    linewidth=1,
    linestyle="--"
)

plt.title(
    "Trade Corridor Momentum vs. Growth Volatility",
    fontsize=15
)

plt.xlabel(
    "Average 3-Year Growth Volatility"
)

plt.ylabel(
    "Average Trade Momentum"
)

plt.tight_layout()

plt.show()

# ===============================================================
# LATEST CORRIDOR MOMENTUM
# ===============================================================

latest_year = master["year"].max()

latest_corridor_data = (
    master[
        master["year"] == latest_year
    ]
    [
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "growth_3yr_avg",
            "growth_3yr_volatility",
            "trade_momentum"
        ]
    ]
    .dropna(
        subset=[
            "trade_momentum"
        ]
    )
)


top_momentum = (
    latest_corridor_data
    .sort_values(
        "trade_momentum",
        ascending=False
    )
    .head(15)
)


plt.figure(figsize=(12, 7))

plt.barh(
    top_momentum["corridor"],
    top_momentum["trade_momentum"]
)

plt.gca().invert_yaxis()

plt.title(
    f"Top 15 Corridors by Trade Momentum in {latest_year}",
    fontsize=15
)

plt.xlabel("Trade Momentum Score")

plt.ylabel("Corridor")

plt.tight_layout()

plt.show()

# ===============================================================
# MOST VOLATILE TRADE CORRIDORS
# ===============================================================

most_volatile = (
    corridor_performance
    .sort_values(
        "avg_volatility",
        ascending=False
    )
    .head(15)
)


plt.figure(figsize=(12, 7))

plt.barh(
    most_volatile["corridor"],
    most_volatile["avg_volatility"]
)

plt.gca().invert_yaxis()

plt.title(
    "Top 15 Most Volatile Trade Corridors",
    fontsize=15
)

plt.xlabel(
    "Average 3-Year Growth Volatility"
)

plt.ylabel("Corridor")

plt.tight_layout()

plt.show()

# ===============================================================
# CORRIDOR PERFORMANCE TABLES
# ===============================================================

print("\nTop corridors by recent momentum:")

display(
    top_momentum[
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "growth_3yr_avg",
            "growth_3yr_volatility",
            "trade_momentum"
        ]
    ]
)


print("\nMost volatile corridors:")

display(
    most_volatile[
        [
            "corridor",
            "avg_growth",
            "avg_momentum",
            "avg_volatility",
            "avg_trade_value"
        ]
    ]
)

# ===============================================================
# BLOCK 4C
# EXTERNAL DRIVERS
# ===============================================================

print("=" * 80)
print("BLOCK 4C - EXTERNAL DRIVERS")
print("=" * 80)


# ===============================================================
# SUPPLY-CHAIN PRESSURE VS TRADE GROWTH
# ===============================================================

pressure_growth = (
    master
    .groupby("year", as_index=False)
    .agg(
        avg_trade_growth=("yoy_growth_pct", "mean"),
        avg_supply_chain_pressure=(
            "avg_supply_chain_pressure",
            "mean"
        )
    )
    .dropna()
)


plt.figure(figsize=(11, 6))

plt.scatter(
    pressure_growth["avg_supply_chain_pressure"],
    pressure_growth["avg_trade_growth"],
    s=90,
    alpha=0.75
)

for _, row in pressure_growth.iterrows():

    if row["year"] in [2008, 2009, 2020, 2021, 2022, 2024]:

        plt.annotate(
            int(row["year"]),
            (
                row["avg_supply_chain_pressure"],
                row["avg_trade_growth"]
            ),
            xytext=(5, 5),
            textcoords="offset points"
        )


plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title(
    "Supply-Chain Pressure vs. Average Trade Growth",
    fontsize=15
)

plt.xlabel("Average Supply-Chain Pressure Index")

plt.ylabel("Average Corridor Trade Growth (%)")

plt.tight_layout()

plt.show()

# ===============================================================
# CORRELATION CHECK
# ===============================================================

pressure_growth_correlation = (
    pressure_growth[
        [
            "avg_supply_chain_pressure",
            "avg_trade_growth"
        ]
    ]
    .corr()
    .iloc[0, 1]
)

print(
    "Correlation between supply-chain pressure "
    f"and average trade growth: "
    f"{pressure_growth_correlation:.3f}"
)


# ===============================================================
# DISRUPTION INTENSITY OVER TIME
# ===============================================================

disruption_timeline = (
    master
    .groupby("year", as_index=False)
    .agg(
        disruption_events=("disruption_events", "mean"),
        disruption_intensity=("disruption_intensity", "mean"),
        avg_trade_impact=("avg_trade_impact", "mean")
    )
)


plt.figure(figsize=(12, 6))

plt.bar(
    disruption_timeline["year"],
    disruption_timeline["disruption_intensity"]
)

plt.title(
    "Global Trade Disruption Intensity by Year",
    fontsize=15
)

plt.xlabel("Year")

plt.ylabel("Disruption Intensity")

plt.xticks(
    disruption_timeline["year"],
    rotation=45
)

plt.tight_layout()

plt.show()

# ===============================================================
# DISRUPTION TRADE IMPACT
# ===============================================================

plt.figure(figsize=(12, 6))

plt.plot(
    disruption_timeline["year"],
    disruption_timeline["avg_trade_impact"],
    linewidth=2.5,
    marker="o"
)

important_years = {
    2001: "9/11",
    2003: "SARS",
    2008: "Financial Crisis",
    2020: "COVID-19",
    2021: "Suez / Capacity Crisis",
    2022: "Russia–Ukraine"
}

for year, label in important_years.items():

    point = disruption_timeline[
        disruption_timeline["year"] == year
    ]

    if not point.empty:

        x = point["year"].iloc[0]
        y = point["avg_trade_impact"].iloc[0]

        plt.annotate(
            label,
            (x, y),
            xytext=(5, 8),
            textcoords="offset points",
            rotation=25
        )


plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title(
    "Estimated Trade-Volume Impact of Disruptions",
    fontsize=15
)

plt.xlabel("Year")

plt.ylabel("Average Trade-Volume Impact (%)")

plt.tight_layout()

plt.show()

# ===============================================================
# EXTERNAL DRIVER SUMMARY
# ===============================================================

external_driver_summary = (
    master
    .groupby("year", as_index=False)
    .agg(
        avg_trade_growth=("yoy_growth_pct", "mean"),
        supply_chain_pressure=(
            "avg_supply_chain_pressure",
            "mean"
        ),
        disruption_events=("disruption_events", "mean"),
        disruption_intensity=("disruption_intensity", "mean"),
        avg_trade_impact=("avg_trade_impact", "mean")
    )
)

display(
    external_driver_summary.tail(10)
)

print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

print(f"""
• Correlation between supply-chain pressure and trade growth:
  {pressure_growth_correlation:.3f}

• Highest supply-chain pressure occurred during 2021.

• Largest tariff activity occurred after 2018.

• COVID-19 generated the largest disruption shock.

• Trade rebounded strongly in 2021 following the 2020 contraction.
""")

# ===============================================================
# BLOCK 5
# CORRIDOR RISK PROFILES
# ===============================================================

print("=" * 80)
print("BLOCK 5 - CORRIDOR RISK PROFILES")
print("=" * 80)


# ===============================================================
# LATEST YEAR DATA
# ===============================================================

latest_year = master["year"].max()

latest_profile = (
    master[
        master["year"] == latest_year
    ]
    [
        [
            "year",
            "corridor",
            "exporter",
            "importer",
            "standard_trade_category",
            "mapped_industry",
            "trade_value_bn_usd",
            "share_global_trade",
            "trade_rank",
            "yoy_growth_pct",
            "growth_3yr_avg",
            "trade_momentum",
            "growth_3yr_volatility",
            "concentration_risk",
            "overall_vulnerability",
            "avg_supply_chain_pressure",
            "disruption_intensity"
        ]
    ]
    .copy()
)


print(
    f"Latest profile year: {latest_year}"
)

print(
    f"Number of corridors: {latest_profile.shape[0]}"
)

# ===============================================================
# DATA-BASED THRESHOLDS
# ===============================================================

thresholds = {
    "importance_median":
        latest_profile["share_global_trade"].median(),

    "volatility_median":
        latest_profile["growth_3yr_volatility"].median(),

    "volatility_high":
        latest_profile["growth_3yr_volatility"].quantile(0.75),

    "concentration_median":
        latest_profile["concentration_risk"].median(),

    "concentration_high":
        latest_profile["concentration_risk"].quantile(0.75),

    "industry_vulnerability_median":
        latest_profile["overall_vulnerability"].median(),

    "industry_vulnerability_high":
        latest_profile["overall_vulnerability"].quantile(0.75),

    "disruption_median":
        latest_profile["disruption_intensity"].median()
}


threshold_table = pd.DataFrame(
    thresholds.items(),
    columns=[
        "Threshold",
        "Value"
    ]
)

display(threshold_table)

# ===============================================================
# GROWTH STATUS
# ===============================================================

latest_profile["growth_status"] = np.select(

    [
        latest_profile["growth_3yr_avg"] > 3,

        latest_profile["growth_3yr_avg"] >= 0,

        latest_profile["growth_3yr_avg"] >= -3
    ],

    [
        "Strong Growth",

        "Moderate Growth",

        "Weak Contraction"
    ],

    default="Severe Contraction"
)


# ===============================================================
# MOMENTUM STATUS
# ===============================================================

latest_profile["momentum_status"] = np.select(

    [
        (
            latest_profile["growth_3yr_avg"] >= 0
        )
        &
        (
            latest_profile["trade_momentum"] > 0
        ),

        (
            latest_profile["growth_3yr_avg"] < 0
        )
        &
        (
            latest_profile["trade_momentum"] > 0
        ),

        (
            latest_profile["growth_3yr_avg"] >= 0
        )
        &
        (
            latest_profile["trade_momentum"] <= 0
        )
    ],

    [
        "Strengthening",

        "Recovering",

        "Growing but Slowing"
    ],

    default="Deteriorating"
)


# ===============================================================
# VOLATILITY RISK
# ===============================================================

latest_profile["volatility_risk"] = np.select(

    [
        latest_profile[
            "growth_3yr_volatility"
        ] >= thresholds["volatility_high"],

        latest_profile[
            "growth_3yr_volatility"
        ] >= thresholds["volatility_median"]
    ],

    [
        "High",

        "Medium"
    ],

    default="Low"
)


# ===============================================================
# CONCENTRATION RISK
# ===============================================================

latest_profile["concentration_risk_level"] = np.select(

    [
        latest_profile[
            "concentration_risk"
        ] >= thresholds["concentration_high"],

        latest_profile[
            "concentration_risk"
        ] >= thresholds["concentration_median"]
    ],

    [
        "High",

        "Medium"
    ],

    default="Low"
)


# ===============================================================
# INDUSTRY VULNERABILITY
# ===============================================================

latest_profile["industry_risk_level"] = np.select(

    [
        latest_profile[
            "overall_vulnerability"
        ] >= thresholds[
            "industry_vulnerability_high"
        ],

        latest_profile[
            "overall_vulnerability"
        ] >= thresholds[
            "industry_vulnerability_median"
        ]
    ],

    [
        "High",

        "Medium"
    ],

    default="Low"
)


print("✓ Individual corridor risk dimensions classified")

# ===============================================================
# RISK FLAGS
# ===============================================================

latest_profile["negative_growth_flag"] = (
    latest_profile["growth_3yr_avg"] < 0
).astype(int)


latest_profile["negative_momentum_flag"] = (
    latest_profile["trade_momentum"] < 0
).astype(int)


latest_profile["high_volatility_flag"] = (
    latest_profile["growth_3yr_volatility"]
    >= thresholds["volatility_high"]
).astype(int)


latest_profile["high_concentration_flag"] = (
    latest_profile["concentration_risk"]
    >= thresholds["concentration_high"]
).astype(int)


latest_profile["high_industry_risk_flag"] = (
    latest_profile["overall_vulnerability"]
    >= thresholds["industry_vulnerability_high"]
).astype(int)


latest_profile["risk_flag_count"] = (
    latest_profile[
        [
            "negative_growth_flag",
            "negative_momentum_flag",
            "high_volatility_flag",
            "high_concentration_flag",
            "high_industry_risk_flag"
        ]
    ]
    .sum(axis=1)
)

# ===============================================================
# OVERALL RISK CATEGORY
# ===============================================================

latest_profile["overall_risk_category"] = pd.cut(

    latest_profile["risk_flag_count"],

    bins=[
        -1,
        0,
        2,
        3,
        5
    ],

    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Critical Risk"
    ]
)


print("✓ Overall risk categories created")

# ===============================================================
# ECONOMIC IMPORTANCE
# ===============================================================

latest_profile["importance_category"] = np.where(

    latest_profile["share_global_trade"]
    >= thresholds["importance_median"],

    "High Importance",

    "Lower Importance"
)


latest_profile["importance_rank"] = (
    latest_profile["trade_rank"]
    .astype(int)
)

# ===============================================================
# RECOMMENDED ACTION
# ===============================================================

conditions = [

    (
        latest_profile["importance_category"]
        == "High Importance"
    )
    &
    (
        latest_profile["overall_risk_category"]
        .isin(
            [
                "High Risk",
                "Critical Risk"
            ]
        )
    ),

    (
        latest_profile["importance_category"]
        == "High Importance"
    )
    &
    (
        latest_profile["overall_risk_category"]
        .isin(
            [
                "Low Risk",
                "Moderate Risk"
            ]
        )
    ),

    (
        latest_profile["importance_category"]
        == "Lower Importance"
    )
    &
    (
        latest_profile["overall_risk_category"]
        .isin(
            [
                "High Risk",
                "Critical Risk"
            ]
        )
    )
]


actions = [

    "Immediate Attention",

    "Protect and Maintain",

    "Monitor or Diversify"
]


latest_profile["recommended_action"] = np.select(

    conditions,

    actions,

    default="Observe"
)


print("✓ Recommended actions assigned")

# ===============================================================
# CORRIDOR RISK PROFILE TABLE
# ===============================================================

risk_profile_columns = [

    "corridor",

    "standard_trade_category",

    "trade_value_bn_usd",

    "importance_rank",

    "importance_category",

    "growth_3yr_avg",

    "trade_momentum",

    "growth_3yr_volatility",

    "growth_status",

    "momentum_status",

    "volatility_risk",

    "concentration_risk_level",

    "industry_risk_level",

    "risk_flag_count",

    "overall_risk_category",

    "recommended_action"

]


risk_profile_table = (

    latest_profile[
        risk_profile_columns
    ]

    .sort_values(

        [
            "risk_flag_count",
            "trade_value_bn_usd"
        ],

        ascending=[
            False,
            False
        ]
    )

    .reset_index(drop=True)

)


display(
    risk_profile_table
)

# ===============================================================
# HIGHEST-PRIORITY CORRIDORS
# ===============================================================

priority_corridors = (

    latest_profile[

        latest_profile["recommended_action"]
        == "Immediate Attention"

    ]

    .sort_values(

        [
            "risk_flag_count",
            "trade_value_bn_usd"
        ],

        ascending=[
            False,
            False
        ]
    )

)


print(
    f"High-importance corridors requiring immediate attention: "
    f"{priority_corridors.shape[0]}"
)


display(

    priority_corridors[

        [
            "corridor",

            "standard_trade_category",

            "trade_value_bn_usd",

            "share_global_trade",

            "growth_3yr_avg",

            "trade_momentum",

            "growth_3yr_volatility",

            "concentration_risk",

            "overall_vulnerability",

            "risk_flag_count",

            "overall_risk_category",

            "recommended_action"
        ]

    ]

)

# ===============================================================
# RISK CATEGORY DISTRIBUTION
# ===============================================================

risk_distribution = (

    latest_profile[
        "overall_risk_category"
    ]

    .value_counts()

    .sort_index()

    .rename("Corridors")

    .to_frame()

)


display(
    risk_distribution
)


plt.figure(figsize=(9, 5))

plt.bar(

    risk_distribution.index.astype(str),

    risk_distribution["Corridors"]

)

plt.title(
    f"Trade Corridor Risk Distribution in {latest_year}",
    fontsize=15
)

plt.xlabel("Risk Category")

plt.ylabel("Number of Corridors")

plt.tight_layout()

plt.show()

# ===============================================================
# LATEST CORRIDOR RISK HEATMAP
# ===============================================================

risk_level_mapping = {

    "Low": 1,

    "Medium": 2,

    "High": 3
}


risk_heatmap_data = (

    latest_profile

    .set_index("corridor")

    [
        [
            "volatility_risk",

            "concentration_risk_level",

            "industry_risk_level"
        ]
    ]

    .replace(
        risk_level_mapping
    )

)


# Add growth and momentum warning indicators

risk_heatmap_data[
    "Negative Growth"
] = (
    latest_profile
    .set_index("corridor")[
        "negative_growth_flag"
    ]
    + 1
)


risk_heatmap_data[
    "Negative Momentum"
] = (
    latest_profile
    .set_index("corridor")[
        "negative_momentum_flag"
    ]
    + 1
)


# Show the 20 corridors with the most warning signals

heatmap_corridors = (

    latest_profile

    .sort_values(

        [
            "risk_flag_count",
            "trade_value_bn_usd"
        ],

        ascending=[
            False,
            False
        ]
    )

    .head(20)

    ["corridor"]

)


risk_heatmap_plot = (
    risk_heatmap_data
    .loc[heatmap_corridors]
)


plt.figure(figsize=(11, 10))

plt.imshow(
    risk_heatmap_plot,
    aspect="auto"
)

plt.colorbar(
    label="Risk Level"
)

plt.xticks(

    ticks=range(
        len(
            risk_heatmap_plot.columns
        )
    ),

    labels=risk_heatmap_plot.columns,

    rotation=45,

    ha="right"
)


plt.yticks(

    ticks=range(
        len(
            risk_heatmap_plot.index
        )
    ),

    labels=risk_heatmap_plot.index
)


plt.title(
    f"Risk Profile of Priority Trade Corridors in {latest_year}",
    fontsize=15
)

plt.xlabel("Risk Dimension")

plt.ylabel("Trade Corridor")

plt.tight_layout()

plt.show()

# ===============================================================
# BLOCK 6
# STRATEGIC PRIORITY MATRIX
# ===============================================================

print("=" * 80)
print("BLOCK 6 - STRATEGIC PRIORITY MATRIX")
print("=" * 80)


# ===============================================================
# 6A. RISK-EXPOSURE PERCENTILES
# ===============================================================

latest_profile["volatility_percentile"] = (
    latest_profile["growth_3yr_volatility"]
    .rank(pct=True)
    * 100
)

latest_profile["concentration_percentile"] = (
    latest_profile["concentration_risk"]
    .rank(pct=True)
    * 100
)

latest_profile["industry_risk_percentile"] = (
    latest_profile["overall_vulnerability"]
    .rank(pct=True)
    * 100
)


latest_profile["relative_risk_exposure"] = (
    latest_profile[
        [
            "volatility_percentile",
            "concentration_percentile",
            "industry_risk_percentile"
        ]
    ]
    .mean(axis=1)
)


print("✓ Relative-risk exposure positions created")


# ===============================================================
# 6B. PERFORMANCE TRAJECTORY
# ===============================================================

latest_profile["performance_trajectory"] = np.select(

    [
        (
            latest_profile["growth_3yr_avg"] >= 0
        )
        &
        (
            latest_profile["trade_momentum"] > 0
        ),

        (
            latest_profile["growth_3yr_avg"] < 0
        )
        &
        (
            latest_profile["trade_momentum"] > 0
        ),

        (
            latest_profile["growth_3yr_avg"] >= 0
        )
        &
        (
            latest_profile["trade_momentum"] <= 0
        )
    ],

    [
        "Strengthening",
        "Recovering",
        "Growing but Slowing"
    ],

    default="Deteriorating"
)


print("✓ Performance trajectories created")


# ===============================================================
# 6C. MATRIX THRESHOLDS
# ===============================================================

importance_threshold = (
    latest_profile["share_global_trade"].mean()
)

risk_exposure_threshold = (
    latest_profile["relative_risk_exposure"].median()
)


print(
    f"Trade-importance threshold: "
    f"{importance_threshold:.3f}"
)

print(
    f"Relative-risk exposure threshold: "
    f"{risk_exposure_threshold:.1f}"
)


# ===============================================================
# 6D. STRATEGIC QUADRANTS
# ===============================================================

latest_profile["priority_quadrant"] = np.select(

    [
        (
            latest_profile["share_global_trade"]
            >= importance_threshold
        )
        &
        (
            latest_profile["relative_risk_exposure"]
            >= risk_exposure_threshold
        ),

        (
            latest_profile["share_global_trade"]
            >= importance_threshold
        )
        &
        (
            latest_profile["relative_risk_exposure"]
            < risk_exposure_threshold
        ),

        (
            latest_profile["share_global_trade"]
            < importance_threshold
        )
        &
        (
            latest_profile["relative_risk_exposure"]
            >= risk_exposure_threshold
        )
    ],

    [
        "Strategic Exposure",
        "Core Stable Corridors",
        "Elevated Exposure"
    ],

    default="Lower-Risk Opportunity"
)


print("✓ Strategic quadrants assigned")


# ===============================================================
# 6E. STRATEGIC PRIORITY TABLE
# ===============================================================

strategic_priority_table = (

    latest_profile[
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "share_global_trade",
            "growth_3yr_avg",
            "trade_momentum",
            "growth_3yr_volatility",
            "concentration_risk",
            "overall_vulnerability",
            "relative_risk_exposure",
            "performance_trajectory",
            "priority_quadrant"
        ]
    ]

    .sort_values(
        [
            "priority_quadrant",
            "share_global_trade",
            "relative_risk_exposure"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )

    .reset_index(drop=True)
)


display(strategic_priority_table)


# ===============================================================
# 6F. QUADRANT DISTRIBUTION
# ===============================================================

quadrant_distribution = (
    latest_profile["priority_quadrant"]
    .value_counts()
    .rename("corridors")
    .to_frame()
)


display(quadrant_distribution)


# ===============================================================
# 6G. STRATEGIC PRIORITY MATRIX
# ===============================================================

trajectory_markers = {
    "Strengthening": "o",
    "Recovering": "^",
    "Growing but Slowing": "s",
    "Deteriorating": "X"
}


plt.figure(figsize=(14, 9))


for trajectory, marker in trajectory_markers.items():

    subset = latest_profile[
        latest_profile["performance_trajectory"]
        == trajectory
    ]

    plt.scatter(
        subset["share_global_trade"],
        subset["relative_risk_exposure"],
        s=(
            subset["trade_value_bn_usd"]
            / latest_profile["trade_value_bn_usd"].max()
            * 1200
            + 60
        ),
        marker=marker,
        alpha=0.70,
        edgecolor="black",
        linewidth=0.6,
        label=trajectory
    )


plt.axvline(
    importance_threshold,
    linestyle="--",
    linewidth=1.2
)

plt.axhline(
    risk_exposure_threshold,
    linestyle="--",
    linewidth=1.2
)


label_corridors = (
    latest_profile
    .nlargest(
        12,
        "share_global_trade"
    )
)


for _, row in label_corridors.iterrows():

    plt.annotate(
        row["corridor"],
        (
            row["share_global_trade"],
            row["relative_risk_exposure"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )


plt.title(
    f"Strategic Trade Corridor Priority Matrix — {latest_year}",
    fontsize=16
)

plt.xlabel(
    "Share of Trade in the 50-Corridor Dataset"
)

plt.ylabel(
    "Relative Risk Exposure Position (Percentile)"
)

plt.legend(
    title="Performance Trajectory",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.show()


# ===============================================================
# 6H. STRATEGIC-EXPOSURE CORRIDORS
# ===============================================================

strategic_exposure_corridors = (

    latest_profile[
        latest_profile["priority_quadrant"]
        == "Strategic Exposure"
    ]

    .sort_values(
        [
            "share_global_trade",
            "relative_risk_exposure"
        ],
        ascending=[
            False,
            False
        ]
    )

    .copy()
)


print(
    "High-importance, high-risk-exposure corridors:",
    strategic_exposure_corridors.shape[0]
)


display(
    strategic_exposure_corridors[
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "share_global_trade",
            "relative_risk_exposure",
            "performance_trajectory",
            "growth_3yr_avg",
            "trade_momentum",
            "priority_quadrant"
        ]
    ]
)


# ===============================================================
# 6I. QUADRANT SUMMARY
# ===============================================================

quadrant_summary = (

    latest_profile

    .groupby(
        "priority_quadrant",
        as_index=False
    )

    .agg(
        corridor_count=("corridor", "count"),

        total_trade_value_bn=(
            "trade_value_bn_usd",
            "sum"
        ),

        average_risk_exposure=(
            "relative_risk_exposure",
            "mean"
        ),

        average_growth=(
            "growth_3yr_avg",
            "mean"
        )
    )

    .sort_values(
        "total_trade_value_bn",
        ascending=False
    )
)

# ===============================================================
# BLOCK 7
# PREDICTIVE MODEL FOR NEXT-YEAR TRADE GROWTH
# ===============================================================

print("=" * 80)
print("BLOCK 7 - PREDICTIVE MODEL FOR NEXT-YEAR TRADE GROWTH")
print("=" * 80)


# ===============================================================
# 7A. DEFINE MODEL FEATURES AND TARGET
# ===============================================================

target_column = "target_next_year_growth"


numeric_features = [

    # Current trade position
    "trade_value_bn_usd",
    "yoy_growth_pct",

    # Historical corridor behavior
    "trade_value_lag1",
    "growth_3yr_avg",
    "growth_3yr_volatility",
    "trade_momentum",

    # Economic importance
    "share_global_trade",

    # Corridor exposure
    "concentration_risk",

    # Shipping environment
    "avg_baltic_dry_index",
    "avg_container_rate_usd_40ft",
    "avg_air_cargo_rate_usd_kg",
    "avg_supply_chain_pressure",
    "avg_on_time_delivery_pct",

    # Commodity environment
    "commodity_price_index",
    "commodity_price_volatility",

    # Disruption environment
    "disruption_count",
    "avg_disruption_severity",
    "disruption_intensity",

    # Industry exposure
    "overall_vulnerability",
    "supplier_concentration_hhi",
    "just_in_time_dependency"
]


categorical_features = [
    "corridor",
    "standard_trade_category",
    "mapped_industry"
]


# Keep only columns that actually exist in master

numeric_features = [
    column
    for column in numeric_features
    if column in master.columns
]

categorical_features = [
    column
    for column in categorical_features
    if column in master.columns
]


model_features = (
    numeric_features
    +
    categorical_features
)


print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal model features:")
print(len(model_features))

# ===============================================================
# 7B. PREPARE MODEL DATA
# ===============================================================

model_data_columns = list(
    dict.fromkeys(
        model_features
        + [
            "year",
            target_column
        ]
    )
)


model_data = (

    master[
        model_data_columns
    ]

    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    .dropna(
        subset=[
            target_column
        ]
    )

    .copy()
)


print(
    "Model-data shape:",
    model_data.shape
)

print(
    "Model years:",
    model_data["year"].min(),
    "-",
    model_data["year"].max()
)

print(
    "Target missing values:",
    model_data[target_column].isna().sum()
)

print(
    "Duplicate column names:",
    model_data.columns.duplicated().sum()
)

# ===============================================================
# 7C. TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ===============================================================

train_data = model_data[
    model_data["year"] <= 2019
].copy()


validation_data = model_data[
    model_data["year"].between(
        2020,
        2021
    )
].copy()


test_data = model_data[
    model_data["year"].between(
        2022,
        2023
    )
].copy()


print(
    "Training observations:",
    train_data.shape[0]
)

print(
    "Validation observations:",
    validation_data.shape[0]
)

print(
    "Test observations:",
    test_data.shape[0]
)


print(
    "\nTraining years:",
    train_data["year"].min(),
    "-",
    train_data["year"].max()
)

print(
    "Validation years:",
    validation_data["year"].min(),
    "-",
    validation_data["year"].max()
)

print(
    "Test years:",
    test_data["year"].min(),
    "-",
    test_data["year"].max()
)

# ===============================================================
# 7D. CREATE X AND y DATASETS
# ===============================================================

X_train = train_data[
    model_features
].copy()

y_train = train_data[
    target_column
].copy()


X_validation = validation_data[
    model_features
].copy()

y_validation = validation_data[
    target_column
].copy()


X_test = test_data[
    model_features
].copy()

y_test = test_data[
    target_column
].copy()

# ===============================================================
# 7E. PREPROCESSING PIPELINE
# ===============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer


numeric_transformer = Pipeline(

    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_transformer = Pipeline(

    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),

        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)


print("✓ Preprocessing pipeline created")

# ===============================================================
# 7F. NAIVE BASELINE MODEL
# ===============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


baseline_validation_predictions = (
    validation_data["yoy_growth_pct"]
    .fillna(
        train_data["yoy_growth_pct"].median()
    )
)


baseline_test_predictions = (
    test_data["yoy_growth_pct"]
    .fillna(
        train_data["yoy_growth_pct"].median()
    )
)


def evaluate_regression(
    actual,
    predicted,
    model_name,
    dataset_name
):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    return {
        "model": model_name,
        "dataset": dataset_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


baseline_validation_results = evaluate_regression(

    y_validation,

    baseline_validation_predictions,

    "Naive Current-Growth Baseline",

    "Validation"
)


baseline_test_results = evaluate_regression(

    y_test,

    baseline_test_predictions,

    "Naive Current-Growth Baseline",

    "Test"
)


display(
    pd.DataFrame(
        [
            baseline_validation_results,
            baseline_test_results
        ]
    )
)

# ===============================================================
# 7G. CANDIDATE MODELS
# ===============================================================

from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)


candidate_models = {

    "Ridge Regression":

        Ridge(
            alpha=10.0
        ),

    "Random Forest":

        RandomForestRegressor(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=4,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":

        GradientBoostingRegressor(
            n_estimators=250,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=5,
            subsample=0.80,
            random_state=42
        )
}


model_pipelines = {}


for model_name, estimator in candidate_models.items():

    model_pipelines[model_name] = Pipeline(

        steps=[

            (
                "preprocessor",
                preprocessor
            ),

            (
                "model",
                estimator
            )
        ]
    )


print("✓ Candidate model pipelines created")

# ===============================================================
# 7H. VALIDATION MODEL COMPARISON
# ===============================================================

validation_results = [

    baseline_validation_results

]


validation_predictions = {}


for model_name, pipeline in model_pipelines.items():

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_validation
    )

    validation_predictions[
        model_name
    ] = predictions

    results = evaluate_regression(

        y_validation,

        predictions,

        model_name,

        "Validation"
    )

    validation_results.append(
        results
    )


validation_results_table = (

    pd.DataFrame(
        validation_results
    )

    .sort_values(
        "MAE"
    )

    .reset_index(drop=True)
)


display(
    validation_results_table.round(3)
)

# ===============================================================
# 7I. SELECT BEST VALIDATION MODEL
# ===============================================================

machine_learning_validation = (

    validation_results_table[

        validation_results_table["model"]
        != "Naive Current-Growth Baseline"

    ]

    .copy()
)


best_model_name = (

    machine_learning_validation

    .sort_values(
        "MAE"
    )

    .iloc[0]["model"]
)


best_validation_mae = (

    machine_learning_validation

    .sort_values(
        "MAE"
    )

    .iloc[0]["MAE"]
)


print(
    "Selected model:",
    best_model_name
)

print(
    f"Validation MAE: "
    f"{best_validation_mae:.3f}"
)

# ===============================================================
# 7J. FINAL MODEL TRAINING
# ===============================================================

development_data = pd.concat(

    [
        train_data,
        validation_data
    ],

    ignore_index=True
)


X_development = development_data[
    model_features
].copy()

y_development = development_data[
    target_column
].copy()


final_model = model_pipelines[
    best_model_name
]


final_model.fit(
    X_development,
    y_development
)


test_predictions = final_model.predict(
    X_test
)


print("✓ Final model trained")

# ===============================================================
# 7K. FINAL TEST RESULTS
# ===============================================================

final_test_results = evaluate_regression(

    y_test,

    test_predictions,

    best_model_name,

    "Test"
)


test_results_table = pd.DataFrame(

    [
        baseline_test_results,
        final_test_results
    ]

)


test_results_table["MAE_improvement_vs_baseline_pct"] = (

    (
        baseline_test_results["MAE"]
        -
        test_results_table["MAE"]
    )

    /

    baseline_test_results["MAE"]

    *

    100
)


display(
    test_results_table.round(3)
)

# ===============================================================
# 7L. TEST-PERIOD PREDICTIONS
# ===============================================================

test_prediction_table = test_data[
    [
        "year",
        "corridor",
        "yoy_growth_pct",
        "growth_3yr_avg",
        "trade_momentum",
        target_column
    ]
].copy()


test_prediction_table[
    "predicted_next_year_growth"
] = test_predictions


test_prediction_table[
    "prediction_error"
] = (

    test_prediction_table[
        target_column
    ]

    -

    test_prediction_table[
        "predicted_next_year_growth"
    ]
)


test_prediction_table[
    "absolute_error"
] = (

    test_prediction_table[
        "prediction_error"
    ]

    .abs()
)


test_prediction_table = (

    test_prediction_table

    .sort_values(
        "absolute_error",
        ascending=False
    )

    .reset_index(drop=True)
)


display(
    test_prediction_table.head(20)
)

# ===============================================================
# 7M. ACTUAL VS PREDICTED
# ===============================================================

plt.figure(
    figsize=(9, 7)
)


plt.scatter(
    y_test,
    test_predictions,
    alpha=0.70,
    edgecolor="black",
    linewidth=0.5
)


plot_min = min(
    y_test.min(),
    test_predictions.min()
)

plot_max = max(
    y_test.max(),
    test_predictions.max()
)


plt.plot(
    [
        plot_min,
        plot_max
    ],
    [
        plot_min,
        plot_max
    ],
    linestyle="--"
)


plt.title(
    f"Actual vs Predicted Next-Year Growth\n{best_model_name}",
    fontsize=15
)

plt.xlabel(
    "Actual Next-Year Growth (%)"
)

plt.ylabel(
    "Predicted Next-Year Growth (%)"
)

plt.tight_layout()

plt.show()

# ===============================================================
# 7N. PREDICTION ERROR DISTRIBUTION
# ===============================================================

plt.figure(
    figsize=(9, 5)
)


plt.hist(
    test_prediction_table[
        "prediction_error"
    ],
    bins=20,
    edgecolor="black"
)


plt.axvline(
    0,
    linestyle="--",
    linewidth=1.2
)


plt.title(
    "Distribution of Prediction Errors",
    fontsize=15
)

plt.xlabel(
    "Actual Growth − Predicted Growth"
)

plt.ylabel(
    "Number of Observations"
)

plt.tight_layout()

plt.show()

# ===============================================================
# 7O. DIRECTIONAL ACCURACY
# ===============================================================

actual_direction = (
    y_test >= 0
)

predicted_direction = (
    test_predictions >= 0
)


directional_accuracy = (

    actual_direction
    ==
    predicted_direction

).mean()


print(
    f"Directional accuracy: "
    f"{directional_accuracy:.1%}"
)

# ===============================================================
# 7P. PERFORMANCE BY TEST YEAR
# ===============================================================

test_year_performance = (

    test_prediction_table

    .groupby(
        "year",
        as_index=False
    )

    .agg(

        observations=(
            "corridor",
            "count"
        ),

        MAE=(
            "absolute_error",
            "mean"
        ),

        RMSE=(
            "prediction_error",
            lambda values:
                np.sqrt(
                    np.mean(
                        values ** 2
                    )
                )
        ),

        actual_average_growth=(
            target_column,
            "mean"
        ),

        predicted_average_growth=(
            "predicted_next_year_growth",
            "mean"
        )
    )
)


display(
    test_year_performance.round(3)
)

# ===============================================================
# 7Q. NEXT-YEAR FORECASTS FROM LATEST OBSERVATIONS
# ===============================================================

latest_forecast_data = (

    master[
        master["year"] == latest_year
    ]

    .copy()
)


X_latest = latest_forecast_data[
    model_features
].copy()


latest_forecast_data[
    "predicted_next_year_growth"
] = final_model.predict(
    X_latest
)


latest_forecast_table = (

    latest_forecast_data[
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "share_global_trade",
            "growth_3yr_avg",
            "trade_momentum",
            "growth_3yr_volatility",
            "concentration_risk",
            "overall_vulnerability",
            "predicted_next_year_growth"
        ]
    ]

    .sort_values(
        "predicted_next_year_growth",
        ascending=False
    )

    .reset_index(drop=True)
)


display(
    latest_forecast_table
)

# ===============================================================
# 7R. FORECAST OUTLOOK CLASSIFICATION
# ===============================================================

latest_forecast_data[
    "forecast_outlook"
] = pd.cut(

    latest_forecast_data[
        "predicted_next_year_growth"
    ],

    bins=[
        -np.inf,
        -3,
        0,
        3,
        np.inf
    ],

    labels=[
        "Significant Contraction",
        "Mild Contraction",
        "Moderate Growth",
        "Strong Growth"
    ]
)


forecast_outlook_distribution = (

    latest_forecast_data[
        "forecast_outlook"
    ]

    .value_counts()

    .sort_index()

    .rename(
        "corridors"
    )

    .to_frame()
)


display(
    forecast_outlook_distribution
)

# ===============================================================
# 7S. WEAKEST FORECAST CORRIDORS
# ===============================================================

weakest_forecast_corridors = (

    latest_forecast_data

    .nsmallest(
        15,
        "predicted_next_year_growth"
    )

    [
        [
            "corridor",
            "standard_trade_category",
            "trade_value_bn_usd",
            "share_global_trade",
            "growth_3yr_avg",
            "trade_momentum",
            "predicted_next_year_growth",
            "forecast_outlook"
        ]
    ]
)


display(
    weakest_forecast_corridors
)

# ===============================================================
# 7T. MERGE FORECASTS INTO LATEST PROFILE
# ===============================================================

forecast_columns = latest_forecast_data[
    [
        "corridor",
        "predicted_next_year_growth",
        "forecast_outlook"
    ]
].copy()


latest_profile = latest_profile.merge(

    forecast_columns,

    on="corridor",

    how="left",

    validate="one_to_one"
)


print(
    "Forecasts merged into latest profile:",
    latest_profile.shape
)


display(
    latest_profile[
        [
            "corridor",
            "share_global_trade",
            "relative_risk_exposure",
            "performance_trajectory",
            "priority_quadrant",
            "predicted_next_year_growth",
            "forecast_outlook"
        ]
    ]

    .sort_values(
        "predicted_next_year_growth"
    )
)

display(quadrant_summary)

# ===============================================================
# BLOCK 8
# EXECUTIVE DECISION SUPPORT
# ===============================================================

print("=" * 80)
print("BLOCK 8 - EXECUTIVE DECISION SUPPORT")
print("=" * 80)


decision_dashboard = (

    latest_profile[

        [

            "corridor",

            "standard_trade_category",

            "mapped_industry",

            "trade_value_bn_usd",

            "share_global_trade",

            "relative_risk_exposure",

            "performance_trajectory",

            "priority_quadrant",

            "predicted_next_year_growth",

            "forecast_outlook"

        ]

    ]

    .copy()

)

recommendations = []


for _, row in decision_dashboard.iterrows():

    quadrant = row["priority_quadrant"]

    outlook = row["forecast_outlook"]


    if quadrant == "Strategic Exposure":

        if outlook in [
            "Significant Contraction",
            "Mild Contraction"
        ]:

            recommendation = (
                "Urgent mitigation and supplier diversification"
            )

        else:

            recommendation = (
                "Strengthen resilience and monitor closely"
            )


    elif quadrant == "Core Lower-Exposure Corridors":

        if outlook in [
            "Moderate Growth",
            "Strong Growth"
        ]:

            recommendation = (
                "Maintain investment and expand trade"
            )

        else:

            recommendation = (
                "Maintain with periodic monitoring"
            )


    elif quadrant == "Elevated Exposure":

        recommendation = (
            "Evaluate diversification opportunities"
        )


    else:

        recommendation = (
            "Explore future growth opportunities"
        )


    recommendations.append(recommendation)


decision_dashboard[
    "executive_recommendation"
] = recommendations

decision_dashboard = (

    decision_dashboard

    .sort_values(

        [

            "share_global_trade",

            "predicted_next_year_growth"

        ],

        ascending=[
            False,
            False
        ]

    )

    .reset_index(drop=True)

)


display(decision_dashboard)

action_summary = (

    decision_dashboard[
        "executive_recommendation"
    ]

    .value_counts()

    .rename("Corridors")

    .to_frame()

)


display(action_summary)

plt.figure(figsize=(14,9))


quadrants = decision_dashboard[
    "priority_quadrant"
].unique()


for quadrant in quadrants:

    subset = decision_dashboard[
        decision_dashboard[
            "priority_quadrant"
        ] == quadrant
    ]

    plt.scatter(

        subset[
            "predicted_next_year_growth"
        ],

        subset[
            "relative_risk_exposure"
        ],

        s=(
            subset[
                "trade_value_bn_usd"
            ]
            /
            decision_dashboard[
                "trade_value_bn_usd"
            ].max()
            *1200
            +50
        ),

        alpha=.70,

        edgecolor="black",

        linewidth=.6,

        label=quadrant

    )


plt.axhline(

    risk_exposure_threshold,

    linestyle="--"

)

plt.axvline(

    0,

    linestyle="--"

)


for _, row in decision_dashboard.nlargest(
    12,
    "trade_value_bn_usd"
).iterrows():

    plt.annotate(

        row["corridor"],

        (

            row["predicted_next_year_growth"],

            row["relative_risk_exposure"]

        ),

        fontsize=8,

        xytext=(5,5),

        textcoords="offset points"

    )


plt.title(
    "Trade Corridor Decision Support Matrix",
    fontsize=16
)

plt.xlabel(
    "Predicted Next-Year Growth (%)"
)

plt.ylabel(
    "Relative Risk Exposure"
)

plt.legend(
    title="Priority Quadrant",
    bbox_to_anchor=(1.02,1)
)

plt.tight_layout()

plt.show()

priority_order = {
    "Strategic Exposure": 0,       # highest priority: important + risky
    "Elevated Exposure": 1,
    "Lower-Risk Opportunity": 2,
    "Core Stable Corridors": 3,    # lowest priority: stable, low risk
}

decision_dashboard["priority_rank"] = (
    decision_dashboard["priority_quadrant"].map(priority_order)
)

top_priorities = (
    decision_dashboard
    .sort_values(
        ["priority_rank", "share_global_trade", "relative_risk_exposure"],
        ascending=[True, False, False]
    )
    .drop(columns="priority_rank")
    .head(10)
)

display(top_priorities)


In [ ]:
# ===============================================================
# BLOCK 7Q
# FEATURE IMPORTANCE ANALYSIS
# ===============================================================

print("=" * 80)
print("BLOCK 7Q - FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

from sklearn.inspection import permutation_importance

# ---------------------------------------------------------------
# BUILT-IN (IMPURITY-BASED) IMPORTANCE FROM THE FITTED MODEL
# ---------------------------------------------------------------
# One-hot encoding splits each categorical feature (e.g. "corridor")
# into many dummy columns, so we roll importances back up to the
# original feature names for an interpretable ranking.

fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_estimator = final_model.named_steps["model"]

encoded_feature_names = fitted_preprocessor.get_feature_names_out()
raw_importances = fitted_estimator.feature_importances_

importance_lookup = pd.DataFrame({
    "encoded_feature": encoded_feature_names,
    "importance": raw_importances
})


def map_to_source_feature(encoded_name):

    if encoded_name.startswith("numeric__"):
        return encoded_name.replace("numeric__", "")

    if encoded_name.startswith("categorical__"):
        stripped = encoded_name.replace("categorical__", "")
        for cat in categorical_features:
            if stripped.startswith(cat + "_"):
                return cat
        return stripped

    return encoded_name


importance_lookup["source_feature"] = (
    importance_lookup["encoded_feature"].apply(map_to_source_feature)
)

builtin_importance = (
    importance_lookup
    .groupby("source_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .rename("builtin_importance")
    .reset_index()
)

print("\nBuilt-in (impurity-based) feature importance:")
display(builtin_importance.round(4))

plt.figure(figsize=(9, 6))
plt.barh(
    builtin_importance["source_feature"][::-1],
    builtin_importance["builtin_importance"][::-1]
)
plt.title(f"Built-in Feature Importance — {best_model_name}", fontsize=14)
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# PERMUTATION IMPORTANCE (OUT-OF-SAMPLE, LESS BIASED TOWARD
# HIGH-CARDINALITY CATEGORICALS THAN IMPURITY-BASED IMPORTANCE)
# ---------------------------------------------------------------

permutation_results = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

permutation_importance_table = (
    pd.DataFrame({
        "feature": X_test.columns,
        "perm_importance_mean": permutation_results.importances_mean,
        "perm_importance_std": permutation_results.importances_std
    })
    .sort_values("perm_importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nPermutation importance (test set, drop in MAE-score when shuffled):")
display(permutation_importance_table.round(4))

plt.figure(figsize=(9, 6))
plt.barh(
    permutation_importance_table["feature"][::-1],
    permutation_importance_table["perm_importance_mean"][::-1],
    xerr=permutation_importance_table["perm_importance_std"][::-1]
)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Permutation Importance (Test Set)", fontsize=14)
plt.xlabel("Mean decrease in validation score when feature is shuffled")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# NOTE ON WHERE THE TWO RANKINGS DISAGREE
# ---------------------------------------------------------------
# Built-in impurity-based importance can be inflated for features the
# tree happened to split on a lot during training, even if that split
# doesn't generalize. Permutation importance measures the actual drop
# in held-out performance, so it is the more trustworthy of the two.
# Features with LOW or NEGATIVE permutation importance despite a HIGH
# built-in score are candidates the model may be latching onto noise,
# rather than a genuinely useful signal.

comparison = (
    builtin_importance
    .merge(
        permutation_importance_table[["feature", "perm_importance_mean"]],
        left_on="source_feature",
        right_on="feature",
        how="inner"
    )
    .drop(columns="feature")
)

comparison["builtin_rank"] = comparison["builtin_importance"].rank(ascending=False)
comparison["perm_rank"] = comparison["perm_importance_mean"].rank(ascending=False)
comparison["rank_gap"] = (comparison["builtin_rank"] - comparison["perm_rank"]).abs()

print("\nFeatures where the two importance methods disagree most:")
display(
    comparison
    .sort_values("rank_gap", ascending=False)
    .round(4)
    .head(8)
)


In [ ]:
# ===============================================================
# BLOCK 7R
# HYPERPARAMETER TUNING CHECK
# ===============================================================

print("=" * 80)
print("BLOCK 7R - HYPERPARAMETER TUNING CHECK")
print("=" * 80)

from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV
from trade_corridor.validation import YearWalkForwardSplit

# Each fold holds out complete calendar years; no year overlaps train/test.
train_sort_order = train_data.sort_values("year").index
X_train_time_sorted = X_train.loc[train_sort_order].reset_index(drop=True)
y_train_time_sorted = y_train.loc[train_sort_order].reset_index(drop=True)
train_years_sorted = train_data.loc[train_sort_order, "year"].reset_index(drop=True)
time_series_cv = list(YearWalkForwardSplit(n_splits=5).split(train_years_sorted))

# ---------------------------------------------------------------
# SEARCH SPACE
# ---------------------------------------------------------------
# Kept deliberately small: with ~1,000 training rows split into 5
# time folds, an aggressive search risks overfitting to whichever
# fold happens to look best, which defeats the purpose of tuning.

gbm_param_distributions = {
    "model__n_estimators": [100, 150, 200, 250, 300, 400],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "model__max_depth": [2, 3, 4],
    "model__min_samples_leaf": [3, 5, 8, 12],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
}

gbm_search = RandomizedSearchCV(
    clone(model_pipelines["Gradient Boosting"]),
    param_distributions=gbm_param_distributions,
    n_iter=25,
    scoring="neg_mean_absolute_error",
    cv=time_series_cv,
    random_state=42,
    n_jobs=-1
)

gbm_search.fit(X_train_time_sorted, y_train_time_sorted)

print("\nBest time-series CV MAE (on training folds):", -gbm_search.best_score_)
print("Best parameters found:")
print(gbm_search.best_params_)

# ---------------------------------------------------------------
# COMPARE TUNED VS. HAND-PICKED MODEL ON THE TRUE VALIDATION SET
# ---------------------------------------------------------------
# model_pipelines["Gradient Boosting"] gets refit later in this
# notebook (Block 7J, on train+validation combined), so we clone it
# fresh here rather than reuse the shared dictionary object -- that
# avoids accidentally evaluating a model that has already seen the
# validation rows it's being scored against.

handpicked_gbm = clone(model_pipelines["Gradient Boosting"])
handpicked_gbm.fit(X_train, y_train)
handpicked_val_predictions = handpicked_gbm.predict(X_validation)
handpicked_val_mae = mean_absolute_error(y_validation, handpicked_val_predictions)

tuned_gbm = gbm_search.best_estimator_
tuned_val_predictions = tuned_gbm.predict(X_validation)
tuned_val_mae = mean_absolute_error(y_validation, tuned_val_predictions)

tuning_comparison = pd.DataFrame([
    {"model": "Hand-picked Gradient Boosting", "validation_MAE": handpicked_val_mae},
    {"model": "Tuned Gradient Boosting", "validation_MAE": tuned_val_mae},
])

print()
display(tuning_comparison.round(3))

if tuned_val_mae < handpicked_val_mae:
    print(
        "\nTuned configuration improves validation MAE -- "
        "consider adopting these hyperparameters for the final model."
    )
else:
    print(
        "\nThe hand-picked configuration already performs at least as well "
        "as the tuned search on the true validation set. Given the small "
        "sample size, this is a reassuring result: the model's performance "
        "isn't fragile to the exact hyperparameters chosen, so we keep the "
        "original configuration for the final model."
    )


In [ ]:
# ===============================================================
# BLOCK 7S
# OUTLIER DIAGNOSTICS ON TEST PREDICTIONS
# ===============================================================

print("=" * 80)
print("BLOCK 7S - OUTLIER DIAGNOSTICS")
print("=" * 80)

# ---------------------------------------------------------------
# STANDARDIZED RESIDUALS
# ---------------------------------------------------------------

diagnostics = test_prediction_table.copy()

residual_mean = diagnostics["prediction_error"].mean()
residual_std = diagnostics["prediction_error"].std()

diagnostics["standardized_residual"] = (
    (diagnostics["prediction_error"] - residual_mean) / residual_std
)

outlier_threshold = 2.0

flagged_outliers = (
    diagnostics
    .loc[diagnostics["standardized_residual"].abs() >= outlier_threshold]
    .sort_values("standardized_residual", key=lambda s: s.abs(), ascending=False)
)

print(
    f"\nFlagged {len(flagged_outliers)} of {len(diagnostics)} test observations "
    f"(|standardized residual| >= {outlier_threshold})"
)

display(
    flagged_outliers[
        [
            "year", "corridor", "yoy_growth_pct", target_column,
            "predicted_next_year_growth", "prediction_error", "standardized_residual"
        ]
    ].round(2)
)

# ---------------------------------------------------------------
# RESIDUAL PLOT WITH OUTLIERS HIGHLIGHTED
# ---------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.scatter(
    diagnostics["predicted_next_year_growth"],
    diagnostics["prediction_error"],
    alpha=0.6, edgecolor="black", linewidth=0.4, label="Test observations"
)

plt.scatter(
    flagged_outliers["predicted_next_year_growth"],
    flagged_outliers["prediction_error"],
    color="red", edgecolor="black", s=90, label="Flagged outlier", zorder=3
)

for _, row in flagged_outliers.iterrows():
    plt.annotate(
        f"{row['corridor']} ({row['year']})",
        (row["predicted_next_year_growth"], row["prediction_error"]),
        fontsize=8, xytext=(6, 6), textcoords="offset points"
    )

plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Prediction Error vs. Predicted Value, with Outliers Flagged", fontsize=14)
plt.xlabel("Predicted Next-Year Growth (%)")
plt.ylabel("Prediction Error (Actual − Predicted)")
plt.legend()
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# METRIC SENSITIVITY: WITH VS. WITHOUT THE FLAGGED OUTLIERS
# ---------------------------------------------------------------
# RMSE and R2 are both sensitive to a small number of extreme errors.
# Reporting metrics with and without the flagged rows shows how much
# of the headline error is being driven by a handful of shock events,
# rather than systematic model weakness.

clean_data = diagnostics.loc[~diagnostics.index.isin(flagged_outliers.index)]

metrics_with_outliers = evaluate_regression(
    diagnostics[target_column],
    diagnostics["predicted_next_year_growth"],
    best_model_name,
    "Test (all observations)"
)

metrics_without_outliers = evaluate_regression(
    clean_data[target_column],
    clean_data["predicted_next_year_growth"],
    best_model_name,
    "Test (outliers excluded)"
)

outlier_sensitivity_table = pd.DataFrame(
    [metrics_with_outliers, metrics_without_outliers]
)

print("\nModel performance with vs. without flagged outliers:")
display(outlier_sensitivity_table.round(3))

print(
    "\nNote: R2 can look worse after removing outliers even though MAE and "
    "RMSE improve. Once the one or two largest-variance rows are dropped, "
    "the remaining test set has very little variance left to explain, so "
    "R2 becomes an unstable and less informative metric on this subset. "
    "MAE and RMSE are the more reliable numbers to quote here."
)


In [ ]:
# ===============================================================
# BLOCK 9
# EMPIRICAL VALIDATION OF THE RISK FRAMEWORK
# ===============================================================

print("=" * 80)
print("BLOCK 9 - EMPIRICAL VALIDATION OF THE RISK FRAMEWORK")
print("=" * 80)

# ---------------------------------------------------------------
# WHY THIS BLOCK EXISTS
# ---------------------------------------------------------------
# The risk_flag_count / priority_quadrant logic in Blocks 5-6 is a
# rule-based scorecard: thresholds are data-driven (percentiles), but
# the decision to combine five binary flags into a single count, and
# the resulting category labels, are analyst judgment calls. That's a
# defensible design (this is how many real-world credit-risk and
# operational-risk scorecards work) but it should be validated, not
# just asserted. Here we check whether corridors flagged as "risky" in
# a given year actually go on to contract in the following year, using
# the historical panel where we already know the outcome.

from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import spearmanr

validation_panel = master.dropna(
    subset=["growth_3yr_volatility", "concentration_risk", "overall_vulnerability"]
).copy()

historical_volatility_high = validation_panel["growth_3yr_volatility"].quantile(0.75)
historical_concentration_high = validation_panel["concentration_risk"].quantile(0.75)
historical_industry_risk_high = validation_panel["overall_vulnerability"].quantile(0.75)

validation_panel["negative_growth_flag"] = (
    validation_panel["growth_3yr_avg"] < 0
).astype(int)

validation_panel["negative_momentum_flag"] = (
    validation_panel["trade_momentum"] < 0
).astype(int)

validation_panel["high_volatility_flag"] = (
    validation_panel["growth_3yr_volatility"] >= historical_volatility_high
).astype(int)

validation_panel["high_concentration_flag"] = (
    validation_panel["concentration_risk"] >= historical_concentration_high
).astype(int)

validation_panel["high_industry_risk_flag"] = (
    validation_panel["overall_vulnerability"] >= historical_industry_risk_high
).astype(int)

validation_panel["risk_flag_count"] = validation_panel[
    [
        "negative_growth_flag", "negative_momentum_flag", "high_volatility_flag",
        "high_concentration_flag", "high_industry_risk_flag"
    ]
].sum(axis=1)

validation_panel = validation_panel.dropna(subset=["target_next_year_growth"])

print(f"\nHistorical corridor-years with a known next-year outcome: {len(validation_panel)}")

# ---------------------------------------------------------------
# DOES A HIGHER RISK FLAG COUNT CORRESPOND TO WORSE NEXT-YEAR GROWTH?
# ---------------------------------------------------------------

growth_by_risk_level = (
    validation_panel
    .groupby("risk_flag_count")["target_next_year_growth"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("\nNext-year growth outcome by risk_flag_count:")
display(growth_by_risk_level)

rank_correlation, p_value = spearmanr(
    validation_panel["risk_flag_count"],
    validation_panel["target_next_year_growth"]
)

print(f"\nSpearman correlation (risk_flag_count vs. next-year growth): "
      f"{rank_correlation:.3f}  (p = {p_value:.4f})")

# ---------------------------------------------------------------
# CAN THE "HIGH / CRITICAL RISK" LABEL PREDICT AN ACTUAL CONTRACTION?
# ---------------------------------------------------------------

validation_panel["flagged_high_or_critical"] = (
    validation_panel["risk_flag_count"] >= 3
).astype(int)

validation_panel["actually_contracted_next_year"] = (
    validation_panel["target_next_year_growth"] < 0
).astype(int)

print("\nConfusion matrix (rows = actual outcome, columns = flagged High/Critical Risk):")
print(
    confusion_matrix(
        validation_panel["actually_contracted_next_year"],
        validation_panel["flagged_high_or_critical"]
    )
)

print(
    classification_report(
        validation_panel["actually_contracted_next_year"],
        validation_panel["flagged_high_or_critical"],
        target_names=["grew", "contracted"],
        zero_division=0
    )
)

# ---------------------------------------------------------------
# INTERPRETATION
# ---------------------------------------------------------------
print(
    "Interpretation: the risk_flag_count is only weakly (and, if positive,\n"
    "in the OPPOSITE direction from a naive 'high risk = will shrink' \n"
    "reading) correlated with actual next-year growth, and the High/Critical\n"
    "Risk label has low precision and recall for predicting an outright\n"
    "contraction. This is an important, honest finding: the risk framework\n"
    "is best presented as a STRUCTURAL VULNERABILITY / EXPOSURE monitor\n"
    "(which corridors are concentrated, volatile, or dependent on\n"
    "vulnerable industries), not as a growth forecasting tool. That job\n"
    "belongs to the Gradient Boosting model in Block 7. The two tools are\n"
    "complementary -- one flags structural exposure, the other predicts\n"
    "near-term direction -- and the report should describe them as such\n"
    "rather than implying the risk score forecasts growth."
)
